In [ ]:
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import StepLR
from PIL import Image
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from tqdm import tqdm
from ultralytics import YOLO
from ultralytics.nn.modules import Detect
from ultralytics.utils.loss import v8DetectionLoss
from types import SimpleNamespace
import copy

# Labels fixos para atributos globais
SCENE_LABELS = ['city street', 'residential', 'highway', 'gas stations', 'parking', 'tunnel', 'bridge', 'railroad', 'roundabout', 'construction', 'parking lot']
WEATHER_LABELS = ['clear', 'rainy', 'foggy', 'snowy', 'overcast', 'undefined', 'partly cloudy']
TIME_LABELS = ['daytime', 'night', 'dawn/dusk', 'undefined']

num_scenes = len(SCENE_LABELS)
num_weather = len(WEATHER_LABELS)
num_timeofday = len(TIME_LABELS)

# Função de perda customizada com pesos ajustados
def custom_loss(det_loss, scene_logits, scene_label, weather_logits, weather_label, time_logits, time_label, scene_weight=1.0, weather_weight=1.0, time_weight=1.0):
    box_loss_val = det_loss[0] if isinstance(det_loss, tuple) else torch.tensor(0.0)
    cls_loss_val = det_loss[1] if isinstance(det_loss, tuple) else torch.tensor(0.0)
    dfl_loss_val = det_loss[2] if isinstance(det_loss, tuple) else torch.tensor(0.0)

    cls_loss_fn = nn.CrossEntropyLoss()
    loss_scene = cls_loss_fn(scene_logits, scene_label) * scene_weight
    loss_weather = cls_loss_fn(weather_logits, weather_label) * weather_weight
    loss_time = cls_loss_fn(time_logits, time_label) * time_weight

    det_total = box_loss_val + cls_loss_val + dfl_loss_val
    total_loss = det_total + loss_scene + loss_weather + loss_time

    return total_loss, {
        'scene': loss_scene.item(),
        'weather': loss_weather.item(),
        'timeofday': loss_time.item(),
        'det_box': box_loss_val.item(),
        'det_cls': cls_loss_val.item(),
        'det_dfl': dfl_loss_val.item()
    }

# Head com atributos globais
class DetectWithAttributes(Detect):
    def __init__(self, nc, ch, f, i=None, dropout_rate=0.3):
        super().__init__(nc=nc, ch=ch)
        self.f = f
        self.i = i
        self.attr_scene = nn.Linear(ch[-1], num_scenes)
        self.attr_weather = nn.Linear(ch[-1], num_weather)
        self.attr_timeofday = nn.Linear(ch[-1], num_timeofday)
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x):
        features = x[-1]
        pooled = self.dropout(features.mean([2, 3]))
        detection_output = super().forward(x)
        return detection_output, self.attr_scene(pooled), self.attr_weather(pooled), self.attr_timeofday(pooled)

# Dataset
class YOLOWithAttributesDataset(Dataset):
    def __init__(self, image_dir, label_dir, attr_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.attr_dir = attr_dir
        self.transform = transform or transforms.ToTensor()
        self.images = sorted([f for f in os.listdir(image_dir) if f.endswith('.jpg')])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.image_dir, img_name)
        label_path = os.path.join(self.label_dir, img_name.replace('.jpg', '.txt'))
        attr_path = os.path.join(self.attr_dir, img_name.replace('.jpg', '.attrs.json'))

        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)

        with open(label_path, 'r') as f:
            boxes = [list(map(float, l.strip().split())) for l in f.readlines()]
        boxes_tensor = torch.tensor(boxes) if boxes else torch.zeros((0, 5))

        with open(attr_path, 'r') as f:
            attr = json.load(f)

        scene_idx = SCENE_LABELS.index(attr['scene']) if attr['scene'] in SCENE_LABELS else 0
        weather_idx = WEATHER_LABELS.index(attr['weather']) if attr['weather'] in WEATHER_LABELS else 0
        time_idx = TIME_LABELS.index(attr['timeofday']) if attr['timeofday'] in TIME_LABELS else 0

        return image, boxes_tensor, scene_idx, weather_idx, time_idx

# Collate
def custom_collate_fn(batch):
    images, boxes, scenes, weathers, times = zip(*batch)
    images = torch.stack(images)
    scenes = torch.tensor(scenes)
    weathers = torch.tensor(weathers)
    times = torch.tensor(times)
    return images, boxes, scenes, weathers, times

# Transformações e paths
base_dir = "/Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/Test/yolo_dataset"
transform = transforms.Compose([
    transforms.Resize((640, 640)),
    transforms.ToTensor(),
])

train_dataset = YOLOWithAttributesDataset(
    os.path.join(base_dir, "images/train"),
    os.path.join(base_dir, "labels/train"),
    os.path.join(base_dir, "attrs/train"),
    transform=transform
)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=custom_collate_fn)

# Modelo
model = YOLO("yolov8n.pt")
detect_layer = model.model.model[-1]
model.model.model[-1] = DetectWithAttributes(
    nc=10,
    ch=[m[0].conv.in_channels for m in detect_layer.cv3],
    f=detect_layer.f,
    i=detect_layer.i
).to("cuda" if torch.cuda.is_available() else "cpu")

# Treinamento
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
det_criterion = v8DetectionLoss(model=model.model)
det_criterion.hyp = SimpleNamespace(box=0.05, cls=0.5, dfl=1.5)

optimizer = torch.optim.Adam(model.model.parameters(), lr=1e-4)
scheduler = StepLR(optimizer, step_size=5, gamma=0.5)

num_epochs = 50
scene_weight = 1.0
weather_weight = 1.0
time_weight = 1.0
best_loss = float("inf")
loss_history = []

for epoch in range(num_epochs):
    model.model.train()
    total_loss = total_scene = total_weather = total_time = total_det = 0

    for imgs, boxes_list, scene_labels, weather_labels, time_labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        imgs = imgs.to(device)
        scene_labels = scene_labels.to(device)
        weather_labels = weather_labels.to(device)
        time_labels = time_labels.to(device)

        targets_list = []
        for b_idx, boxes in enumerate(boxes_list):
            if boxes.numel() > 0:
                for box in boxes:
                    targets_list.append(torch.cat([torch.tensor([b_idx], device=device), box.to(device)]))
        targets_tensor = torch.stack(targets_list) if targets_list else torch.empty((0, 6), device=device)

        optimizer.zero_grad()
        det_feats, scene_out, weather_out, time_out = model.model(imgs)

        if targets_tensor.shape[0] > 0:
            batch_dict = {
                'img': imgs,
                'batch_idx': targets_tensor[:, 0].long(),
                'cls': targets_tensor[:, 1].long(),
                'bboxes': targets_tensor[:, 2:]
            }
            det_loss, _ = det_criterion(det_feats, batch_dict)
        else:
            zero = torch.tensor(0.0, requires_grad=True).to(device)
            det_loss = (zero, zero, zero)

        total, parts = custom_loss(det_loss, scene_out, scene_labels, weather_out, weather_labels, time_out, time_labels, scene_weight, weather_weight, time_weight)
        total.backward()
        optimizer.step()

        total_loss += total.item()
        total_det += parts['det_box'] + parts['det_cls'] + parts['det_dfl']
        total_scene += parts['scene']
        total_weather += parts['weather']
        total_time += parts['timeofday']

    scheduler.step()

    loss_history.append((total_loss, total_det, total_scene, total_weather, total_time))
    print(f"[Epoch {epoch+1}] Total: {total_loss:.2f} | Det: {total_det:.2f} | Scene: {total_scene:.2f} | Weather: {total_weather:.2f} | Time: {total_time:.2f}")

    if total_loss < best_loss:
        best_loss = total_loss
        best_model_state = copy.deepcopy(model.model.state_dict())
        torch.save(best_model_state, os.path.join(base_dir, "yolov8_with_attributes_best.pt"))
        print("✅ Novo melhor modelo salvo!")

# Visualização
epochs = list(range(1, num_epochs + 1))
total, det, scene, weather, timeofday = zip(*loss_history)
plt.plot(epochs, total, label="Total")
plt.plot(epochs, det, label="Detecção")
plt.plot(epochs, scene, label="Scene")
plt.plot(epochs, weather, label="Weather")
plt.plot(epochs, timeofday, label="Time of Day")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss por Época")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import os
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from ultralytics import YOLO
from ultralytics.nn.modules import Detect
from ultralytics.utils.ops import non_max_suppression

# Labels
CLASS_NAMES = ["traffic sign", "traffic light", "car", "rider", "motor",
               "person", "bus", "truck", "bike", "train"]
SCENE_LABELS = ['city street', 'residential', 'highway', 'gas stations', 'parking', 'tunnel',
                'bridge', 'railroad', 'roundabout', 'construction', 'parking lot']
WEATHER_LABELS = ['clear', 'rainy', 'foggy', 'snowy', 'overcast', 'undefined', 'partly cloudy']
TIME_LABELS = ['daytime', 'night', 'dawn/dusk', 'undefined']

# Caminhos
val_images_dir = "/Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/Test/yolo_dataset/images/validation"
model_path = "/Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/Test/yolo_dataset/yolov8_with_attributes_best.pt"

# Transformação de entrada
transform = transforms.Compose([
    transforms.Resize((640, 640)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Head com atributos globais
class DetectWithAttributes(Detect):
    def __init__(self, nc, ch, f, i=None):
        super().__init__(nc=nc, ch=ch)
        self.f = f
        self.i = i
        self.attr_scene = torch.nn.Linear(ch[-1], len(SCENE_LABELS))
        self.attr_weather = torch.nn.Linear(ch[-1], len(WEATHER_LABELS))
        self.attr_timeofday = torch.nn.Linear(ch[-1], len(TIME_LABELS))
        self.dropout = torch.nn.Dropout(p=0.3)

    def forward(self, x):
        features = x[-1]
        pooled = self.dropout(features.mean([2, 3]))
        detection_output = super().forward(x)
        return detection_output, self.attr_scene(pooled), self.attr_weather(pooled), self.attr_timeofday(pooled)

# Carregar modelo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = YOLO("yolov8n.pt")
detect_layer = model.model.model[-1]
model.model.model[-1] = DetectWithAttributes(
    nc=len(CLASS_NAMES),
    ch=[m[0].conv.in_channels for m in detect_layer.cv3],
    f=detect_layer.f,
    i=detect_layer.i
).to(device)
model.model.load_state_dict(torch.load(model_path, map_location=device))
model.model.eval()

# Função de inferência
def run_inference(image_path):
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        preds_raw, scene_logits, weather_logits, time_logits = model.model(image_tensor)

    scene_pred = SCENE_LABELS[scene_logits.argmax(1).item()]
    weather_pred = WEATHER_LABELS[weather_logits.argmax(1).item()]
    time_pred = TIME_LABELS[time_logits.argmax(1).item()]

    # Aplicar NMS
    preds = non_max_suppression(preds_raw, conf_thres=0.3, iou_thres=0.5)[0]

    # Visualização com OpenCV
    image_cv = np.array(image)
    image_cv = cv2.cvtColor(image_cv, cv2.COLOR_RGB2BGR)
    h, w = image_cv.shape[:2]

    if preds is not None and len(preds) > 0:
        for det in preds.cpu():
            x1, y1, x2, y2, conf, class_id = det
            x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])
            label = CLASS_NAMES[int(class_id)]
            cv2.rectangle(image_cv, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(image_cv, f"{label} {conf:.2f}", (x1, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # Adicionar atributos preditos no rodapé
    footer = f"Scene: {scene_pred} | Weather: {weather_pred} | Time: {time_pred}"
    cv2.putText(image_cv, footer, (10, h - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 255), 2)

    # Mostrar resultado
    image_rgb = cv2.cvtColor(image_cv, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 6))
    plt.imshow(image_rgb)
    plt.axis('off')
    plt.title("Predição YOLOv8 com Atributos")
    plt.tight_layout()
    plt.show()

# Rodar em várias imagens de validação
val_images = sorted(os.listdir(val_images_dir))
for i in range(0, min(50, len(val_images))):  # Altere esse valor para testar mais
    print(f"\n🖼️ Imagem {i+1}: {val_images[i]}")
    run_inference(os.path.join(val_images_dir, val_images[i]))
